# Part 1: Exchange rates from the World Bank API

We use the Data360 API to get annual exchange rates for 2015–2025. The indicator is `WB_WDI_PA_NUS_FCRF`: local currency units per 1 US dollar, averaged over the year.


In [1]:
from pathlib import Path
import requests
import pandas as pd

url = "https://data360api.worldbank.org/data360/data"
params = {
    "DATABASE_ID": "WB_WDI",
    "INDICATOR": "WB_WDI_PA_NUS_FCRF",
    "timePeriodFrom": 2015,
    "timePeriodTo": 2025,
    "FREQ": "A"
}
csv_file = Path("data/exchange_rates_2015_2025.csv")


## Download

The API sends at most 1,000 rows at a time. We use `skip` to get the next page. A timeout or HTTP error stops the download so we do not save an incomplete CSV.


In [2]:
def get_page(skip):
    try:
        response = requests.get(url, params={**params, "skip": skip}, timeout=30)
        response.raise_for_status()
        result = response.json()
    except (requests.RequestException, ValueError) as error:
        raise RuntimeError(f"API request failed at skip={skip}: {error}") from error

    if not isinstance(result, dict) or "count" not in result or not isinstance(result.get("value"), list):
        raise ValueError("Unexpected API response")
    return result


In [3]:
download_again = False

if csv_file.exists() and not download_again:
    rates = pd.read_csv(csv_file)
    print("Loaded saved CSV")
else:
    first_page = get_page(0)
    all_rows = first_page["value"]
    total = first_page["count"]

    while len(all_rows) < total:
        next_page = get_page(len(all_rows))
        if not next_page["value"]:
            raise ValueError("The API returned an empty page before the download was complete")
        all_rows.extend(next_page["value"])

    if len(all_rows) != total:
        raise ValueError("Number of downloaded rows does not match the API total")

    raw = pd.DataFrame(all_rows)
    rates = raw[["TIME_PERIOD", "REF_AREA", "OBS_VALUE"]].rename(columns={
        "TIME_PERIOD": "year",
        "REF_AREA": "country_code",
        "OBS_VALUE": "lcu_per_usd"
    })
    rates["year"] = pd.to_numeric(rates["year"], errors="coerce")
    rates["lcu_per_usd"] = pd.to_numeric(rates["lcu_per_usd"], errors="coerce")
    rates = rates.dropna().drop_duplicates(["year", "country_code"])
    rates["year"] = rates["year"].astype(int)
    rates = rates.sort_values(["country_code", "year"]).reset_index(drop=True)

    csv_file.parent.mkdir(exist_ok=True)
    rates.to_csv(csv_file, index=False)
    print(f"Saved {len(rates)} rows to {csv_file}")


Loaded saved CSV


## Check the result


In [4]:
print("Rows:", len(rates))
print("Countries and areas:", rates["country_code"].nunique())
print("Missing values:", rates.isna().sum().to_dict())
print("Duplicate country-year rows:", rates.duplicated(["country_code", "year"]).sum())
print("Rows by year:")
print(rates.groupby("year").size())
print("Kazakhstan:")
display(rates[rates["country_code"] == "KAZ"])


Rows: 2226
Countries and areas: 213
Missing values: {'year': 0, 'country_code': 0, 'lcu_per_usd': 0}
Duplicate country-year rows: 0
Rows by year:
year
2015    210
2016    210
2017    210
2018    208
2019    208
2020    209
2021    207
2022    207
2023    205
2024    199
2025    153
dtype: int64
Kazakhstan:


,year,country_code,lcu_per_usd
1050,2015,KAZ,221.728333
1051,2016,KAZ,342.160000
1052,2017,KAZ,326.001023
1053,2018,KAZ,344.705833
1054,2019,KAZ,382.747311
1055,2020,KAZ,412.953333
1056,2021,KAZ,425.907500
1057,2022,KAZ,460.165000
1058,2023,KAZ,456.165000
1059,2024,KAZ,468.962500


## Note for the report

I downloaded the annual official exchange rate indicator from the World Bank Data360 API for 2015–2025. The values show the average number of local currency units per US dollar. I collected all pages of the API result and saved the cleaned country-year data in a CSV file. Some countries do not have a value for every year, and 2025 has fewer observations than the earlier years. These are annual averages, not daily exchange rates.
